In [ ]:
# Proyecto Final CCI - AndesVision AI
C1 = 1141318021   # integrante 1
C2 = 1018421230   # integrante 2 (Juan Sebastián García)
S = C1 + C2
print("S =", S)

# Proyecto Final CCI - AndesVision AI

Ciencia Computacional Intermedia. Docente: Ph.D. Ricardo Andrés Fonseca Perdomo.
Integrantes: Juan Pablo Vásquez, Juan Sebastián García.

El código de cada parte está en la carpeta `cci/` (`parte_a.py` a `parte_g.py`). Este notebook llama esas funciones en orden y muestra los resultados principales. Las figuras se guardan en `resultados/` y las tablas completas en `resultados/tablas/`.

Para reproducir los resultados se debe ejecutar todo en orden (Run All), porque todas las simulaciones usan el mismo generador `rng = np.random.default_rng(S)`.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from IPython.display import display

from cci.config import parametros_del_grupo, preparar_rutas, guardar_tabla
from cci import parte_a, parte_b, parte_c, parte_d, parte_e, parte_f, parte_g, resumen

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 20)

P = parametros_del_grupo(S)
print(P)
RUTA_DATOS, RUTA_RESULTADOS, RUTA_G = preparar_rutas()
rng = np.random.default_rng(S)
params = {}     # parámetros que se van guardando para el simulador de la parte G

# Parte A. Auditoría y preparación de los datos

## A1. Auditoría de `solicitudes.csv`

Cada regla sale de cómo funciona el sistema real (sección 4.1 del enunciado):

| Regla física | Por qué |
|---|---|
| Cada `id_solicitud` es único | Una solicitud se registra una vez |
| Los tiempos entre llegadas son ≥ 0 | El tiempo no retrocede |
| `red_ms` > 0 y `tam_kb` en un rango posible | Tiempos y tamaños no pueden ser negativos ni absurdos |
| Si `cache_hit = 1`, entonces `gpu_ms = 0` | Una respuesta de caché no usa GPU |
| `latencia_ms` ≥ `red_ms` + `gpu_ms` | Porque L = red + espera + G y la espera en cola es ≥ 0 |
| Si `exito = 0`, entonces `intentos = 3` | El SDK solo abandona tras 3 intentos fallidos |

In [ ]:
# A1: auditoría
crudo = parte_a.leer_solicitudes(RUTA_DATOS)
A1 = parte_a.a1_auditoria(crudo)
display(A1["tabla_A1"])

In [ ]:
# A1: limpieza
A1L = parte_a.a1_limpiar(crudo)
df = A1L["df"]
display(A1L["antes_despues"].round(3))

La conversión de tamaño es una hipótesis de unidad observacional. Se cuantifica su sensibilidad omitiendo tamaños dudosos. Las mediciones faltantes se conservan como ausentes y se excluyen por variable.

## A2. Regímenes de operación

Se prueba si toda la ventana se puede ver como un solo proceso de Poisson. Como no, se buscan los cambios de tasa por máxima verosimilitud y se elige el número de cambios con AIC y BIC.

In [ ]:
# A2: cambios de tasa
A2 = parte_a.a2_detectar_cambios(df)
display(A2["modelos"].round(1))
display(A2["tabla_tramos"][["tramo", "desde (s)", "hasta (s)", "llegadas", "tasa (sol/s)", "p dispersión 10s",
                            "2·ganancia de un cambio extra", "penalización BIC"]])

In [ ]:
# A2: régimen estable
REG = parte_a.a2_definir_regimenes(df, A2)
df_estable, dur_estable = REG["df_estable"], REG["dur_estable"]
parte_a.exportar_limpio(df, RUTA_RESULTADOS)
display(REG["comparacion"].round(4))

**Selección de régimen.** Se comparan AIC y BIC, la sensibilidad a la rejilla y la similitud de los segmentos exteriores. El régimen de referencia agrupa los dos segmentos exteriores y excluye el pico.

## A3. Estadística descriptiva por régimen y versión

In [ ]:
# A3: descriptivos (la tabla completa, con n y RIC, queda en resultados/tablas/A3_descriptivos.csv)
A3 = parte_a.a3_descriptivos(df)
display(A3["tabla_A3_resumen"].round(2))

In [ ]:
parte_a.figura_1(df, A2, RUTA_RESULTADOS)

**Figura 1.** La banda central marca el pico inferido por verosimilitud. El gráfico muestra cómo cambian la tasa y la latencia en ese intervalo.

# Parte B. Llegadas, fallos y reintentos

## B1. Proceso de llegadas

Las llegadas se modelan como un **proceso de Poisson homogéneo** con tasa $\lambda$.
Para evitar sesgos por la exclusión del pico, los tiempos entre llegadas $\Delta t_i = t_i - t_{i-1}$ se calculan **estrictamente dentro de cada segmento continuo** (`estable_1` y `estable_2`).

La tasa estimada es $\hat{\lambda} = 1/\overline{\Delta t}$, y el intervalo de confianza al nivel $1-\alpha$ del grupo es:
$$\hat{\lambda} \pm z_{1-\alpha/2} \frac{\hat{\lambda}}{\sqrt{n}}$$

In [ ]:
# B1
B1 = parte_b.b1_llegadas(df, df_estable, A2["SEGMENTOS"], P, rng)

## B2. Bloques de 100 solicitudes
El modelo binomial supone una probabilidad de reintento constante dentro de cada régimen.

In [ ]:
# B2
B2 = parte_b.b2_bloques(df, rng)
display(B2["tabla_b2"].round(4))

## B3. Estimador de máxima verosimilitud de `intentos`

El SDK abandona después del tercer fallo, así que con 3 intentos no se sabe cómo terminó el último. La deducción del estimador y de su varianza va en las hojas manuscritas. Aquí se calcula $\hat p$ con esa fórmula, su intervalo con la información de Fisher y una simulación que lo compara con el estimador que sale del promedio de intentos.

In [ ]:
# B3
B3 = parte_b.b3_intentos(df_estable, P, rng)

## B4. Probabilidad en una ventana de 10 s

En una ventana de 10 s, las solicitudes totales siguen $\text{Poisson}(10 \hat{\lambda})$.
Por la **propiedad de adelgazamiento (Poisson thinning)**, como cada solicitud requiere reintento de forma independiente con probabilidad $\hat{p}$, el conteo de solicitudes con reintento es:
$$Z \sim \text{Poisson}(\hat{\mu} = 10 \hat{\lambda} \hat{p})$$

Con el parámetro $r$ del grupo:
$$k = \lceil \hat{\mu} \rceil + r$$
$$P(Z \ge k) = 1 - \sum_{j=0}^{k-1} e^{-\hat{\mu}} \frac{\hat{\mu}^j}{j!}$$
Se verifica analíticamente y por Montecarlo con $200\,000$ réplicas independientes.

In [ ]:
# B4
B4 = parte_b.b4_ventana_10s(B1["lambda_hat"], B3["p_mle"], P, rng)

# Parte C. Tiempo de cómputo y de red

## C1. Distribución de G

G es el tiempo de GPU de cualquier solicitud. Con probabilidad $h$ la respuesta sale de la caché y $G = 0$; si no, va a la GPU y $G > 0$ es continua. Por eso G es una variable mixta: tiene una masa en 0 y una densidad en $(0, \infty)$. La deducción de su CDF, $E[G]$ y $\mathrm{Var}(G)$ va en las hojas manuscritas; en el notebook se comparan esas fórmulas con los datos.

In [ ]:
# C1
C1 = parte_c.c1_masa_en_cero(df_estable, P, params)

## C2. Máxima verosimilitud para la parte continua

**Log-verosimilitud Gamma** con $x_1,\dots,x_n>0$:
$$\ell(k,\theta)=(k-1)\sum\log x_i-\frac{1}{\theta}\sum x_i-nk\log\theta-n\log\Gamma(k)$$
**Log-verosimilitud Lognormal:**
$$\ell(\mu,s)=-\sum\log x_i-n\log(s\sqrt{2\pi})-\frac{1}{2s^2}\sum(\log x_i-\mu)^2$$
Se maximizan numéricamente con `scipy.optimize.minimize`, que minimiza $-\ell$. Los parámetros se optimizan en escala logarítmica para que siempre sean positivos. Los errores estándar salen del **hessiano observado**: $\mathrm{Var}(\hat\theta)\approx H^{-1}$.

In [ ]:
# C2: ajuste Gamma y Lognormal
C2 = parte_c.c2_emv(C1["G_c"], P)
display(C2["tabla_C2"].round(4))

In [ ]:
# C2: bondad de ajuste
C2_ajuste = parte_c.c2_bondad_ajuste(C1["G_c"], C2, rng)

### Gamma marginal contra Gamma condicionada al tamaño

In [ ]:
# C2: modelo condicional
C2_cond = parte_c.c2_gamma_condicional(C1["validas"], C2, C2_ajuste, P, params)

In [ ]:
parte_c.figura_2(C1, C2, RUTA_RESULTADOS)

**Figura 2.** El salto de la CDF en 0 es la proporción de respuestas desde caché. La parte continua es asimétrica a la derecha y ninguna distribución única la describe bien en la cola; el modelo condicionado al tamaño ajusta mucho mejor.

In [ ]:
# C1: E[G] y Var(G) con las fórmulas de la variable mixta
C1M = parte_c.c1_momentos(C1, C2, P)
display(C1M["tabla_C1"].round(2))

## C3. ¿El tiempo de red es normal?

In [ ]:
# C3
C3n = parte_c.c3_normalidad(df_estable)
C3g = parte_c.c3_gamma_desplazada(C3n, params)
C3_colas = parte_c.c3_colas(C3n, C3g, P)
display(C3_colas["tabla_C3"].round(5))

In [ ]:
parte_c.figura_3(C3n, C3g, P, RUTA_RESULTADOS)

## C4. Tiempo total de GPU de N solicitudes no cacheadas

Sea $T_N=\sum_{i=1}^N G_i$ y el umbral $u=N(\hat m+15)$. Equivale a que el **promedio** de las N pase de $\hat m+15$ ms. Se usan tres aproximaciones analíticas:
1. **TLC (normal):** $P\approx 1-\Phi\!\left(\frac{15\sqrt N}{\hat\sigma}\right)$.
2. **Gamma exacta:** si $G_i\sim\text{Gamma}(k,\theta)$ son independientes, $T_N\sim\text{Gamma}(Nk,\theta)$. Se usan los parámetros de momentos, que reproducen la media y la varianza observadas.
3. **Edgeworth (TLC corregido por asimetría $\gamma$):** $P\approx 1-\Phi(z)+\varphi(z)\,\frac{\gamma}{6\sqrt N}(z^2-1)$.

Se comparan con dos simulaciones: (a) remuestreo de las G observadas y (b) bloques **reales** de N solicitudes consecutivas. La (b) también prueba el supuesto de independencia.

In [ ]:
# C4
C4 = parte_c.c4_suma_gpu(C1["G_c"], df_estable, C2, P, rng)
display(C4["tabla_C4"][["TLC", "Gamma exacta", "Edgeworth", "Simulación (remuestreo)", "Bloques reales"]].round(5))

# Parte D. Dependencias y esperanza condicional

## D1. Tamaño de la imagen y caché

In [ ]:
# D1
D1 = parte_d.d1_tamano_cache(df_estable, P, params)
display(D1["conjunta"].round(4), D1["cond_cache"].round(4))

In [ ]:
parte_d.figura_4(D1, P, RUTA_RESULTADOS)

## D2. Esperanza condicional y leyes de esperanza y varianza total

In [ ]:
# D2
D2 = parte_d.d2_esperanza_condicional(df_estable, P)
display(D2["tabla_medias"].round(3), D2["tabla_D2"].round(4))

La tabla verifica ambas identidades usando los mismos casos completos por grupos.

## D3. Regresión de G sobre el tamaño

Se ajusta $G = \beta_0 + \beta_1 \cdot$ tamaño resolviendo las ecuaciones normales con matrices. La deducción va en las hojas manuscritas; la primera celda imprime las sumas que se usan en ella.

In [ ]:
# D3: mínimos cuadrados
D3_mco = parte_d.d3_mco(C1["validas"], P)

In [ ]:
# D3: residuos y corrección
D3_res = parte_d.d3_residuos(D3_mco)
D3_corr = parte_d.d3_correccion(D3_mco, C2_cond, P, params)
display(D3_corr["tabla_D3"].round(5))

In [ ]:
parte_d.figura_5(D3_mco, D3_res, D3_corr, D2, rng, RUTA_RESULTADOS)

## D4. Covarianza y correlación entre tamaño y tiempo de GPU

In [ ]:
# D4
D4 = parte_d.d4_correlacion(D3_mco, D2, P)
parametros_CD = parte_d.exportar_parametros(P, params, C1, C2, C1M, D3_mco, D3_corr, RUTA_RESULTADOS)

# Parte E. Monitoreo y decisión de alertas

In [ ]:
mon = parte_e.leer_monitoreo(RUTA_DATOS, P)

## E1. Sensibilidad y tasa de falsos positivos
Sensibilidad = P(alerta | incidente). Tasa de FP = P(alerta | sin incidente)

In [ ]:
# E1
E1 = parte_e.e1_sensibilidad(mon, P)
display(E1["e1"][["sens", "sens_lo", "sens_hi", "fpr", "fpr_lo", "fpr_hi"]].round(4))

## E2. Probabilidad de incidente dada una alerta en producción
$$\text{VPP} = \frac{\text{sens}\cdot\pi}{\text{sens}\cdot\pi + \text{FPR}\cdot(1-\pi)}$$
Con la π de producción (pequeña) el VPP es mucho menor que en `monitoreo.csv`, porque ahí los incidentes se inyectaron a propósito y son mucho más frecuentes (efecto de la tasa base).
El IC del VPP se obtiene propagando por Montecarlo la incertidumbre de la sensibilidad y la FPR, que se sortean de sus distribuciones Beta de Jeffreys: Beta(x + ½, n − x + ½).

In [ ]:
# E2
E2 = parte_e.e2_vpp(mon, E1, P, rng)
display(E2["e2"].round(4))

## E3. ¿D1 y D2 son condicionalmente independientes?
Dentro de cada estado (con y sin incidente) se arma la tabla 2×2 D1×D2 y se prueba la independencia con la prueba exacta de Fisher.
Si fueran independientes: P(D1=1, D2=1 | estado) = P(D1=1 | estado)·P(D2=1 | estado).
Efecto en E2: se compara el VPP de «D1 y D2» medido directamente con el que daría suponer independencia.

In [ ]:
# E3
E3 = parte_e.e3_independencia(mon, E1, E2, P)
display(E3["tabla_E3"].round(4))

## E4. Falsas alarmas por día
En un día hay 1440 minutos; los minutos sin incidente son ≈ 1440·(1−π).
Si cada uno da falsa alarma con probabilidad FPR, de forma independiente: X ~ Binomial(n₀, FPR).
Para propagar la incertidumbre se sortea la FPR de su Beta y se calcula P(X ≤ kmax) en cada sorteo. Criterio conservador: la regla cumple si el percentil α de esa probabilidad (el peor caso razonable) sigue siendo ≥ 0.90.

In [ ]:
# E4
E4 = parte_e.e4_falsas_alarmas(E1, P, rng, params)
display(E4["e4"].round(4))

In [ ]:
parte_e.figura_6(E4, P, RUTA_RESULTADOS)

# Parte F. Inferencia y comparación de versiones

## F1. Distribución muestral de la media de la latencia
**Población:** las latencias del régimen estable después de la limpieza. Es la mejor representación disponible del sistema en operación normal; el muestreo con reemplazo equivale a una población infinita con esa distribución.
Para cada n se extraen M muestras y se analiza la forma de la distribución de x̄.
**Cobertura efectiva:** porcentaje de veces que x̄ ± z·s/√n contiene la media verdadera.

In [ ]:
# F1
F1 = parte_f.f1_distribucion_muestral(df_estable, P, rng)
display(F1["tabla_f1"].round(4))

In [ ]:
parte_f.figura_7(F1, RUTA_RESULTADOS)

## F2. IC para la latencia media
El IC t clásico supone (1) observaciones independientes y (2) normalidad o n grande.
(2) se cumple por el TLC (F1). (1) probablemente no: hay cola, y si una solicitud espera, la siguiente también tiende a esperar, de modo que las latencias están autocorrelacionadas.
Corrección: **medias por lotes** (*batch means*). Se ordena en el tiempo, se parte en k lotes consecutivos y el IC se construye con las k medias, que son casi independientes entre sí.

In [ ]:
# F2
F2 = parte_f.f2_ic_latencia(df_estable, P)

## F3. ¿La versión B es mejor que la A?
Tres comparaciones, todas como B − A:
- (i) tiempo medio de GPU de las solicitudes no cacheadas → Welch
- (ii) P(intentos > 1) → Newcombe (Wilson)
- (iii) P(latencia > τ) → Newcombe (Wilson)

Como se reportan juntas, se usa **Bonferroni**: cada IC al nivel 1 − α/3, para que las tres a la vez tengan confianza conjunta ≥ 1 − α.

In [ ]:
# F3
F3 = parte_f.f3_comparar_versiones(df_estable, P, params)
display(F3["tabla_F3"].round(4))

## F4. Cobertura de los intervalos de Wald y Wilson
Para cada (n, p verdadera) se generan 10⁴ conteos X ~ Binomial(n, p) y se mide qué porcentaje de intervalos contiene p. El nivel nominal es `conf`.

In [ ]:
# F4
F4 = parte_f.f4_cobertura(P, rng, params)
display(F4["tabla_f4"].pivot(index="p", columns="n", values=["Wald", "Wilson"]).round(3))

In [ ]:
parte_f.figura_8(F4, P, RUTA_RESULTADOS)

## F5. Tamaño de muestra para un semiancho de 1 ms

Se busca el número de solicitudes por versión para que el intervalo de F3(i) tenga un semiancho de máximo 1 ms, con el mismo nivel de Bonferroni. La deducción va en las hojas manuscritas; aquí se calcula n y se comprueba por simulación.

In [ ]:
# F5
F5 = parte_f.f5_tamano_muestra(F3, P, rng)

## F6. Validación con `solicitudes_validacion.csv`
Se aplica la misma limpieza de A1 y se compara cada parámetro con su IC original.
Si el valor de validación cae fuera del IC original, o la prueba de dos muestras rechaza, el parámetro cambió.

In [ ]:
# F6
F6 = parte_f.f6_validacion(RUTA_DATOS, df_estable, dur_estable, P)
display(F6["tabla_F6"].round(4))

# Parte G. Simulación y decisión de capacidad

El simulador de la cola está programado en la función `simular` de `cci/parte_g.py`, sin librerías de eventos discretos.

In [ ]:
# ajuste de los parámetros del simulador
G0 = parte_g.g0_ajustar(df_estable, dur_estable)

## G1. Validación con dos GPUs

Se simula el sistema actual (2 GPUs, régimen estable) y se comparan los cuantiles y la proporción de latencias sobre τ con los datos reales.

In [ ]:
# G1
G1 = parte_g.g1_validacion(df_estable, G0, P, rng)
display(G1["valid_g"].round(4))

In [ ]:
parte_g.figura_g1(G1, RUTA_G)

## G2. Número de GPUs necesario con el tráfico proyectado

Con la tasa de llegadas multiplicada por g se estima P(L > τ) para 1 a 5 GPUs. c* es el menor número de GPUs cuyo límite superior del intervalo queda por debajo de ε.

In [ ]:
# G2
G2 = parte_g.g2_capacidad(G0, G1, P, rng, RUTA_G)
display(G2["g2"].round(5))

In [ ]:
parte_g.figura_9(G2, P, RUTA_G)

## G3. Incertidumbre de c* y comparación con M/M/c

Se repite la elección de c* con remuestras bootstrap de los datos para ver qué tan estable es, y se compara con la fórmula de una cola M/M/c.

In [ ]:
# G3
G3 = parte_g.g3_incertidumbre(df_estable, dur_estable, G0, G1, G2, P, rng)
display(G3["g2"].round(5))

# Resumen y tablas completas

Se guardan las cifras principales en `resultados/resumen_consolidado.json` y las tablas completas en `resultados/tablas/` para usarlas en el informe.

In [ ]:
resultados = dict(crudo=crudo, mon=mon, A1=A1, A1L=A1L, A2=A2, REG=REG, B1=B1, B2=B2, B3=B3, B4=B4,
                  C1=C1, C2=C2, C2_ajuste=C2_ajuste, C3_colas=C3_colas, C4=C4,
                  D1=D1, D2=D2, D3_mco=D3_mco, D3_corr=D3_corr, D4=D4, E1=E1, E4=E4,
                  F1=F1, F2=F2, F3=F3, F5=F5, F6=F6, G1=G1, G2=G2, G3=G3)
resumen_final = resumen.construir_resumen(resultados, P, RUTA_RESULTADOS)

tablas = {
    "A1_auditoria": A1["tabla_A1"], "A2_dispersion": A2["tabla_dispersion"], "A2_modelos": A2["modelos"],
    "A2_tramos": A2["tabla_tramos"], "A2_resolucion": A2["tabla_resolucion"], "A2_comparacion": REG["comparacion"],
    "A2_margen": REG["tabla_margen"], "A3_descriptivos": A3["tabla_A3"], "B2_bloques": B2["tabla_b2"],
    "C2_parametros": C2["tabla_C2"], "C3_colas": C3_colas["tabla_C3"], "C4_aproximaciones": C4["tabla_C4"],
    "D1_conjunta": D1["conjunta"], "D1_condicionales": D1["cond_cache"], "D2_medias": D2["tabla_medias"],
    "D2_descomposicion": D2["tabla_D2"], "D3_coeficientes": D3_corr["tabla_D3"], "E1_sensibilidad": E1["e1"],
    "E2_vpp": E2["e2"], "E3_independencia": E3["tabla_E3"], "E4_falsas_alarmas": E4["e4"], "F1_muestral": F1["tabla_f1"],
    "F3_versiones": F3["tabla_F3"], "F4_cobertura": F4["tabla_f4"], "F6_validacion": F6["tabla_F6"],
    "G0_reintentos": G0["evid"], "G1_validacion": G1["valid_g"], "G2_G3_capacidad": G3["g2"],
}
for nombre, tabla in tablas.items():
    guardar_tabla(tabla, nombre, RUTA_RESULTADOS)
print(f"{len(tablas)} tablas guardadas en resultados/tablas/")